# C.4 找得到也選得中嗎？

# C：推理步驟、驗證與測試時算力

前置：下一字生成、SFT與標準答案驗證。像解數學題：寫出步驟、多試幾種辦法、檢查答案，是三個不同動作。真實推理質量需要trained模型；默認例子用可驗證的數字，避免把長文字當思考保證。




In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：多試幾次，還要知道怎麼選**

候選包含正解，不代表最後答案已經選中。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/reasoning.svg")))

**先想一想：**[3,3,4]對2+2，多數決會選對嗎？

多數決偏向最常出現的答案，verifier按規則查正誤。正確答案可能少數；反之錯誤verifier也會選錯，兩個階段都要測。

**把直覺寫成數學：**coverage與selection_accuracy分開；oracle verifier只適用於已知可驗證任務。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
from collections import Counter

candidates = [3, 3, 4]
majority = Counter(candidates).most_common(1)[0][0]
verified = next((x for x in candidates if x == 2 + 2), None)
print("多數", majority, "算術驗證", verified)
assert majority != 4 and verified == 4

**如何讀結果：**這裏有精確標準答案，開放事實不能假設同樣完美verifier。

**只改一件事：**只移除正確候選4，檢查verifier應返回None而非編造。
